# Advanced Analysis

Deeper investigation into zone archetypes, mobile payment trends, driver shift optimization, trip efficiency, and Tableau export CSVs.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv('chicago_taxi_2024_cleaned.csv')

In [47]:
# O'Hare trips — do tip patterns differ by time of day?
ohare = df[df["Pickup Community Area"] == 76]
ohare.groupby("Time of Day")[["Fare", "Tip Pct", "Trip Miles", "Speed MPH"]].mean().round(2)

,Fare,Tip Pct,Trip Miles,Speed MPH
Time of Day,,,,
Afternoon,40.98,17.55,14.48,22.43
Evening,39.67,17.91,14.36,24.41
Morning,41.65,16.99,14.37,24.32
Night,37.07,16.55,13.51,32.98


In [48]:
# Zero extras vs trips with extras — fare difference?
df["Has Extras"] = (df["Extras"] > 0).astype(int)
df["Has Tolls"] = (df["Tolls"] > 0).astype(int)

df.groupby(["Has Extras", "Has Tolls"])[["Fare", "Trip Total", "Trip Miles", "Tip Pct"]].mean().round(2)

Fare  Trip Total  Trip Miles  Tip Pct
Has Extras Has Tolls                                        
0          0          18.43       20.20        4.63    10.98
           1          33.60       43.15       11.79    15.34
1          0          30.61       42.12       10.81    17.41
           1          46.07       77.60       17.95    19.45

In [49]:
# Payment type share month over month — is Mobile growing?
payment_monthly = df.groupby(["Trip Start Month", "Payment Type"])["Trip ID"].count().reset_index()
payment_monthly.columns = ["Month", "Payment Type", "Trips"]
total_monthly = payment_monthly.groupby("Month")["Trips"].transform("sum")
payment_monthly["Share Pct"] = (payment_monthly["Trips"] / total_monthly * 100).round(2)

payment_monthly[payment_monthly["Payment Type"].isin(["Credit Card", "Mobile", "Cash"])].pivot(
    index="Month", columns="Payment Type", values="Share Pct"
)

Payment Type,Cash,Credit Card,Mobile
Month,,,
2024-01,28.33,37.75,15.47
2024-02,27.76,38.22,14.74
2024-03,27.64,38.83,16.07
2024-04,27.03,39.78,17.18
2024-05,26.42,41.75,17.59
2024-06,26.37,42.22,18.16
2024-07,28.82,40.11,15.12
2024-08,28.25,41.17,14.97
2024-09,27.53,41.89,14.89


In [50]:
# Best day + hour combo for drivers — avg fare and tip combined
df.groupby(["Day of Week", "Trip Start Hour"]).agg(
    trips=("Trip ID", "count"),
    avg_fare=("Fare", "mean"),
    avg_tip=("Tips", "mean"),
    avg_earnings=("Trip Total", "mean")
).round(2).sort_values("avg_earnings", ascending=False).head(20)

trips  avg_fare  avg_tip  avg_earnings
Day of Week Trip Start Hour                                        
Monday      0                14603     30.75     4.35         40.25
Tuesday     0                12615     30.36     4.05         39.85
            1                 7167     30.22     3.70         39.69
Monday      1                 8651     30.23     3.84         39.43
Sunday      20               36661     30.12     4.67         39.35
            23               18745     30.11     4.15         38.81
            21               30238     29.77     4.44         38.67
Wednesday   0                11692     29.32     3.90         38.45
Saturday    5                 6050     32.16     3.44         37.97
Wednesday   1                 6189     28.86     3.56         37.96
Friday      4                 5413     33.12     3.08         37.83
            5                10251     32.24     3.61         37.78
Sunday      19               40338     28.79     4.43         37.51
Tuesday     2                 3730     29.18     2.99         37.48
Sunday      22               24109     29.07     4.11         37.40
Monday      23               16698     29.05     3.85         37.29
Wednesday   4                 4680     33.01     2.55         37.07
Monday      4                 5532     32.62     2.63         36.94
Thursday    5                 9764     31.48     3.47         36.86
Monday      5                10701     31.29     3.42         36.80

In [51]:
# Are short quick trips more time-efficient than long trips for drivers?
df.groupby("Trip Type").agg(
    avg_fare_per_minute=("Fare", lambda x: (x / df.loc[x.index, "Trip Minutes"].replace(0, pd.NA)).mean()),
    avg_trip_minutes=("Trip Minutes", "mean"),
    avg_fare=("Fare", "mean"),
    total_trips=("Trip ID", "count")
).round(2)

,avg_fare_per_minute,avg_trip_minutes,avg_fare,total_trips
Trip Type,,,,
Long,1.23,35.83,39.60,2386584
Medium,0.99,17.38,15.33,1434603
Short,23.49,9.04,11.36,2513026


In [52]:
# Taxi loyalty — does the same taxi repeatedly serve the same zone?
loyalty = df.groupby(["Taxi ID", "Pickup Community Area"]).size().reset_index(name="trip_count")
loyalty_strong = loyalty[loyalty["trip_count"] >= 50].sort_values("trip_count", ascending=False)
print(f"Taxi-zone pairs with 50+ trips: {len(loyalty_strong):,}")
loyalty_strong.head(20)

Taxi-zone pairs with 50+ trips: 18,623


,Taxi ID,Pickup Community Area,trip_count
10923,1d16af0a709e55f205152e890a0968800553ecaf1e643b...,32,3978
9166,18ef2c0a775680d8a4821ce443e80f5f7a98e7d1fc1b69...,28,3740
105234,fe3f654450472e6aa241347b9ebb7f78ce4628a0069cfc...,32,3216
97483,ecbd0442d6b871f22f17122917c37190e9aaf098687b39...,32,3210
12833,218100bdb8cbbbde4b4edf363cf5890c6bb7e3cb749f7f...,32,3153
57950,8da9e1d18757022c6a6a614fc2d38483e38aae441feff5...,8,3056
96816,ea9c7f865233f880e5f00abb728092901eeaf52c85a8c1...,8,3049
89955,da4802fcc8e9eb10ee00b1212582430f9a6af7bc2982f9...,32,3012
1007,020531ad67ef679261206ce9876298202b29824371fcf9...,8,2876
80544,c3bb697ebd10de692888e34f4f3c1ad35d7d5bc0bccc15...,32,2875


In [53]:
# Top 20 pickup zones with avg fare and time of day breakdown
zone_stats = df.groupby("Pickup Community Area").agg(
    total_trips=("Trip ID", "count"),
    avg_fare=("Fare", "mean"),
    avg_miles=("Trip Miles", "mean"),
    avg_tip_pct=("Tip Pct", "mean")
).round(2).sort_values("total_trips", ascending=False).head(20)
print(zone_stats)

                       total_trips  avg_fare  avg_miles  avg_tip_pct
Pickup Community Area                                               
76                         1369845     39.93      14.23        17.36
8                          1340946     14.47       3.50        14.80
32                         1016339     14.32       3.47        16.70
28                          592082     13.50       3.27        14.34
33                          221150     20.14       5.28        13.58
56                          217343     34.72      11.27        16.12
6                           199952     17.52       4.57        11.87
7                           127814     16.26       4.16        12.70
3                            96974     18.94       5.52         5.85
77                           74959     20.07       5.97         5.17
24                           57486     19.59       5.02        10.67
41                           52355     22.49       6.68         5.89
2                            46313

In [54]:
# What time of day are each top zone's trips concentrated in?
top_zones = [76, 8, 32, 28, 33, 56, 6, 7, 3, 77]
df[df["Pickup Community Area"].isin(top_zones)].groupby(
    ["Pickup Community Area", "Time of Day"]
)["Trip ID"].count().unstack("Time of Day").fillna(0).astype(int)

Time of Day,Afternoon,Evening,Morning,Night
Pickup Community Area,,,,
3,32131,15863,39002,9978
6,58694,37894,74213,29151
7,37408,29411,43176,17819
8,467108,333159,369807,170872
28,169883,107143,256081,58975
32,364235,279800,285295,87009
33,96539,52246,57677,14688
56,66171,54419,47491,49262
76,418610,425723,255972,269540


In [55]:
# Weekend vs weekday for top zones — which areas are nightlife driven?
df[df["Pickup Community Area"].isin(top_zones)].groupby(
    ["Pickup Community Area", "Is Weekend"]
)["Trip ID"].count().unstack("Is Weekend").rename(columns={0: "Weekday", 1: "Weekend"})

Is Weekend,Weekday,Weekend
Pickup Community Area,,
3,73925,23049
6,148128,51824
7,93345,34469
8,1032534,308412
28,500670,91412
32,844876,171463
33,161790,59360
56,164693,52650
76,1030246,339599


In [ ]:
# 1. Monthly Revenue
monthly = (
    df.groupby('Trip Start Month')
    .agg(
        Total_Trips   = ('Trip Total', 'count'),
        Total_Revenue = ('Trip Total', 'sum'),
        Avg_Fare      = ('Fare',       'mean'),
        Avg_Tip_Pct   = ('Tip Pct',    'mean')
    )
    .reset_index()
    .rename(columns={'Trip Start Month': 'Month'})
)
monthly['Total_Revenue'] = monthly['Total_Revenue'].round(2)
monthly['Avg_Fare']      = monthly['Avg_Fare'].round(2)
monthly['Avg_Tip_Pct']   = monthly['Avg_Tip_Pct'].round(2)
monthly.to_csv('monthly_revenue.csv', index=False)

# 2. Hourly Heatmap
heatmap = (
    df.groupby(['Day of Week', 'Trip Start Hour'])
    .agg(
        Trip_Count = ('Trip Total', 'count'),
        Avg_Fare   = ('Fare',       'mean')
    )
    .reset_index()
)
heatmap['Avg_Fare'] = heatmap['Avg_Fare'].round(2)
heatmap.to_csv('hourly_heatmap.csv', index=False)

# 3. Company Scorecard
company = (
    df.groupby('Company')
    .agg(
        Trip_Count  = ('Trip Total', 'count'),
        Avg_Fare    = ('Fare',       'mean'),
        Avg_Tip_Pct = ('Tip Pct',    'mean'),
        Total_Rev   = ('Trip Total', 'sum')
    )
    .reset_index()
)
company['Market_Share_Pct'] = (company['Trip_Count'] / company['Trip_Count'].sum() * 100).round(2)
company['Avg_Fare']         = company['Avg_Fare'].round(2)
company['Avg_Tip_Pct']      = company['Avg_Tip_Pct'].round(2)
company['Total_Rev']        = company['Total_Rev'].round(2)
company = company.sort_values('Trip_Count', ascending=False).head(15)
company.to_csv('company_scorecard.csv', index=False)

# 4. Payment Monthly Mix
pay_monthly = (
    df.groupby(['Trip Start Month', 'Payment Type'])
    .agg(Trip_Count = ('Trip Total', 'count'))
    .reset_index()
    .rename(columns={'Trip Start Month': 'Month', 'Payment Type': 'Payment_Type'})
)
totals = pay_monthly.groupby('Month')['Trip_Count'].transform('sum')
pay_monthly['Share_Pct'] = (pay_monthly['Trip_Count'] / totals * 100).round(2)
pay_monthly.to_csv('payment_monthly.csv', index=False)

# 5. Tip Rate by Payment × Time of Day
tip_matrix = (
    df.groupby(['Payment Type', 'Time of Day'])
    .agg(
        Avg_Tip_Pct = ('Tip Pct',    'mean'),
        Trip_Count  = ('Trip Total', 'count')
    )
    .reset_index()
    .rename(columns={'Payment Type': 'Payment_Type', 'Time of Day': 'Time_of_Day'})
)
tip_matrix['Avg_Tip_Pct'] = tip_matrix['Avg_Tip_Pct'].round(2)
tip_matrix.to_csv('tip_payment_time.csv', index=False)

# 6. Trip Type Economics
trip_econ = (
    df.groupby('Trip Type')
    .agg(
        Trip_Count    = ('Trip Total',    'count'),
        Avg_Fare      = ('Fare',          'mean'),
        Avg_Fare_Mile = ('Fare Per Mile', 'mean'),
        Avg_Tip_Pct   = ('Tip Pct',       'mean'),
        Avg_Speed     = ('Speed MPH',     'mean'),
        Avg_Total     = ('Trip Total',    'mean')
    )
    .reset_index()
    .rename(columns={'Trip Type': 'Trip_Type'})
)
for col in ['Avg_Fare','Avg_Fare_Mile','Avg_Tip_Pct','Avg_Speed','Avg_Total']:
    trip_econ[col] = trip_econ[col].round(2)
trip_econ.to_csv('trip_type_economics.csv', index=False)

# 7. Time of Day Summary 
tod = (
    df.groupby('Time of Day')
    .agg(
        Trip_Count  = ('Trip Total', 'count'),
        Avg_Fare    = ('Fare',       'mean'),
        Avg_Tip_Pct = ('Tip Pct',    'mean')
    )
    .reset_index()
    .rename(columns={'Time of Day': 'Time_of_Day'})
)
tod['Avg_Fare']    = tod['Avg_Fare'].round(2)
tod['Avg_Tip_Pct'] = tod['Avg_Tip_Pct'].round(2)
tod.to_csv('time_of_day_summary.csv', index=False)

# 8. Zone Summary
zone = (
    df.groupby('Pickup Community Area')
    .agg(
        Trip_Count  = ('Trip Total', 'count'),
        Avg_Fare    = ('Fare',       'mean'),
        Avg_Tip_Pct = ('Tip Pct',    'mean'),
        Total_Rev   = ('Trip Total', 'sum')
    )
    .reset_index()
    .rename(columns={'Pickup Community Area': 'Community_Area'})
)
zone['Avg_Fare']    = zone['Avg_Fare'].round(2)
zone['Avg_Tip_Pct'] = zone['Avg_Tip_Pct'].round(2)
zone['Total_Rev']   = zone['Total_Rev'].round(2)

archetype_map = {
    76: 'Airport', 56: 'Airport',
    32: 'Business', 28: 'Business',
     8: 'Nightlife',  6: 'Nightlife', 7: 'Nightlife',
    33: 'Tourism',   77: 'Tourism'
}
zone['Archetype'] = zone['Community_Area'].map(archetype_map).fillna('Other')
zone = zone.dropna(subset=['Community_Area'])
zone['Community_Area'] = zone['Community_Area'].astype(int)
zone.to_csv('zone_summary.csv', index=False)

In [58]:
# Chicago community area centroids (lat/lon)
coords = {
    1:  (41.9773, -87.6696), 2:  (41.9957, -87.6890), 3:  (41.9994, -87.6693),
    4:  (41.9750, -87.6519), 5:  (41.9558, -87.6441), 6:  (41.9430, -87.6538),
    7:  (41.9251, -87.6516), 8:  (41.9003, -87.6319), 9:  (41.8534, -87.6185),
    10: (41.8697, -87.5879), 11: (41.8349, -87.6056), 12: (41.8654, -87.6561),
    13: (41.8677, -87.6315), 14: (41.8511, -87.6386), 15: (41.8283, -87.6334),
    16: (41.8106, -87.6250), 17: (41.8314, -87.6561), 18: (41.8014, -87.6530),
    19: (41.7855, -87.6023), 20: (41.7744, -87.5959), 21: (41.7528, -87.5994),
    22: (41.7470, -87.6319), 23: (41.7631, -87.6530), 24: (41.7474, -87.6696),
    25: (41.8447, -87.6957), 26: (41.8954, -87.6957), 27: (41.8845, -87.7189),
    28: (41.8754, -87.6957), 29: (41.8558, -87.7189), 30: (41.8314, -87.7189),
    31: (41.8754, -87.7442), 32: (41.8827, -87.6319), 33: (41.8447, -87.6185),
    34: (41.8106, -87.6957), 35: (41.9251, -87.7189), 36: (41.9430, -87.7189),
    37: (41.9558, -87.7189), 38: (41.9750, -87.7189), 39: (41.9957, -87.7189),
    40: (42.0133, -87.7189), 41: (42.0133, -87.6957), 42: (41.9957, -87.6538),
    43: (41.9430, -87.6319), 44: (41.9251, -87.6319), 45: (41.9251, -87.6957),
    46: (41.9430, -87.6957), 47: (41.9558, -87.6957), 48: (41.9750, -87.6957),
    49: (41.9558, -87.6319), 50: (41.9750, -87.6319), 51: (41.8697, -87.7189),
    52: (41.8511, -87.7189), 53: (41.8283, -87.7189), 54: (41.8106, -87.7189),
    55: (41.7855, -87.7189), 56: (41.7855, -87.7442), 57: (41.7631, -87.7189),
    58: (41.7474, -87.7189), 59: (41.7528, -87.6696), 60: (41.7744, -87.6696),
    61: (41.7855, -87.6696), 62: (41.8014, -87.6957), 63: (41.8283, -87.6957),
    64: (41.9003, -87.7189), 65: (41.9003, -87.6957), 66: (41.9003, -87.6696),
    67: (41.9003, -87.6538), 68: (41.9251, -87.6538), 69: (41.9430, -87.6538),
    70: (41.9558, -87.6538), 71: (41.9750, -87.6538), 72: (41.9957, -87.6319),
    73: (42.0133, -87.6696), 74: (42.0133, -87.6319), 75: (41.9957, -87.6957),
    76: (41.9808, -87.9088), 77: (41.9957, -87.6319)
}

zone = pd.read_csv('zone_summary.csv')
zone['Latitude']  = zone['Community_Area'].map(lambda x: coords.get(x, (None, None))[0])
zone['Longitude'] = zone['Community_Area'].map(lambda x: coords.get(x, (None, None))[1])
zone = zone.dropna(subset=['Latitude', 'Longitude'])
zone.to_csv('zone_summary.csv', index=False)
print(f"✓ zone_summary.csv updated with coordinates — {len(zone)} zones")

✓ zone_summary.csv updated with coordinates — 77 zones


In [61]:
print(df["Trip Start Month"].dtype)
print(df["Trip Start Month"].head(5))
print()
print(df["Trip Start Hour"].dtype)
print(df["Trip Start Hour"].head(5))
print()
payment = df["Payment Type"].value_counts().reset_index()
print(payment.columns.tolist())
print(payment.head())

str
0    2024-12
1    2024-12
2    2024-12
3    2024-12
4    2024-12
Name: Trip Start Month, dtype: str

int32
0    23
1    23
2    23
3    23
4    23
Name: Trip Start Hour, dtype: int32

['Payment Type', 'count']
  Payment Type    count
0  Credit Card  2539659
1         Cash  1738219
2       Mobile  1033570
3       Prcard   751262
4      Unknown   256575
